# TRENDYOL E-TİCARET YARIŞMASI 2026 — ÇÖZÜM NOTEBOOK'U
## Arama–Ürün Alaka Sınıflandırması | Nihai Public LB: 0.805

Bu notebook, çözümün gelişim yolculuğunu ve nihai modeli sunar.

## 0. GİRİŞ VE PROBLEM
**Görev:** Bir (arama terimi, ürün) çiftinin ALAKALI (1) mı ALAKASIZ (0) mı olduğunu tahmin etmek. **Metrik:** macro-averaged F1.

**Zorluk:** Eğitim verisi SADECE pozitif (alakalı) çiftler içerir; negatif örnekleri yarışmacı üretmek zorundadır.

**Yolculuk özeti:**
1. TF-IDF + elle özellikler + LightGBM ............... 0.704 → 0.743
2. Araç değişimi: BGE cross-encoder reranker ......... 0.743 → 0.805

**Nihai çözüm:** `bge-reranker-v2-m3` (sıfır-atış) + eşik -1.5

In [1]:
import gc
import numpy as np
import pandas as pd
import polars as pl
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print("✅ Kütüphaneler yüklendi.")


✅ Kütüphaneler yüklendi.


In [2]:
# Not: Kolon isimleri veri sürümüne göre değişebildiği için, otomatik bir
# çözücü ile kanonik şemaya eşliyoruz (boru hattını dayınıklı kılar).

import os, glob

COMPETITION_SLUG = "trendyol-e-ticaret-yarismasi-2026-kaggle"

def find_base_dir(filename="terms.csv", root="/kaggle/input"):
    """Kaggle'ın veriyi hangi klasöre mount ettiğini otomatik bulur.
    Sabit '/kaggle/input/competitions/<slug>/' varsayımı yerine gerçek konumu arar,
    çünkü yarışma verisi genelde '/kaggle/input/<slug>/' altına mount edilir
    (aradaki 'competitions/' segmenti genelde YOKTUR)."""
    matches = glob.glob(f"{root}/**/{filename}", recursive=True)
    if not matches:
        available = os.listdir(root) if os.path.isdir(root) else "KLASÖR YOK (/kaggle/input hiç oluşmamış)"
        raise FileNotFoundError(
            f"'{filename}' hiçbir yerde bulunamadı.\n"
            f"1) Kaggle notebook editöründe sağ panelden 'Add Input' ile "
            f"'{COMPETITION_SLUG}' yarışma verisini eklediğinizden emin olun.\n"
            f"2) '/kaggle/input' içeriği: {available}"
        )
    return os.path.dirname(matches[0])

BASE = find_base_dir("terms.csv")

PATHS = {
    "train":      f"{BASE}/training_pairs.csv",
    "items":      f"{BASE}/items.csv",
    "terms":      f"{BASE}/terms.csv",
    "submission": f"{BASE}/submission_pairs.csv",
}
ALIASES = {
    "TERM_ID":  ["term_id", "termid", "query_id"],
    "ITEM_ID":  ["item_id", "itemid", "product_id"],
    "QUERY":    ["query", "search_term", "term", "keyword"],
    "TITLE":    ["title", "product_title", "product_name", "name"],
    "CATEGORY": ["category_name", "leaf_category", "category", "categories"],
    "BRAND":    ["brand_name", "brand", "marka"],
}
def resolve(df):
    low = {c.lower(): c for c in df.columns}
    out = {}
    for canon, opts in ALIASES.items():
        for o in opts:
            if o in low:
                out[canon] = low[o]; break
    return out
print(f"✅ Config hazır. BASE = {BASE}")


✅ Config hazır. BASE = /kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle


In [3]:
df_terms = pl.read_csv(PATHS["terms"])
df_items = pl.read_csv(PATHS["items"])
CFG = {**resolve(df_terms), **resolve(df_items)}
# term/item id'lerini standart isimlere getir
df_terms = df_terms.rename({CFG["TERM_ID"]: "term_id"}) if CFG["TERM_ID"] != "term_id" else df_terms
df_items = df_items.rename({CFG["ITEM_ID"]: "item_id"}) if CFG["ITEM_ID"] != "item_id" else df_items
CFG["TERM_ID"], CFG["ITEM_ID"] = "term_id", "item_id"
print(f"✅ Veri okundu — terms: {df_terms.height:,} | items: {df_items.height:,}")


✅ Veri okundu — terms: 50,153 | items: 962,873


## 4. EDA — ÖNEMLİ BULGULAR

- Eğitim verisi: 250.000 satır, TAMAMI pozitif (label=1).
- Test seti: 3.359.679 çift (eğitimin ~13 katı) — hesaplama yükü burada.
- **KRİTİK KEŞİF:** Kendi ürettiğimiz negatifleri bir LLM (Qwen2.5-7B) ile denetlediğimizde, bu "negatiflerin" ~%60'ının aslında ALAKALI (yanlış etiketli) olduğunu gördük. Kategori-içi karıştırma, aynı kategoriden ürün seçtiği için sıklıkla gerçekten alakalı çiftler üretiyordu. → Bu, ilk-nesil modelin neden "fazla cömert" kaldığını açıklar.
- Gerçek test dağılımı (LLM etiketlemesinden): ~%63 alakasız / %37 alakalı.

## 5. YAKLAŞIM 1 — TF-IDF + ELLE ÖZELLİKLER + LightGBM (0.704 → 0.743)

İlk nesilde, metinsel benzerlik sinyallerini elle çıkarıp LightGBM ile sınıflandırdık. Kullanılan başlıca özellikler:

- Kelime ve KARAKTER n-gram TF-IDF benzerliği (Türkçe eklere dayanıklı)
- Token örtüşmesi, jaccard, sorgu/başlık uzunlukları
- MARKA uyumu (+1/-1/0) ve CİNSİYET uyumu (+1/-1/0)
- Kategori token örtüşmesi, öznitelik değer örtüşmesi
- Negatifler: kategori-içi (zor) + kategoriler-arası (kolay) karıştırma.

**SONUÇ:** Bu yaklaşım 0.743'te platoya ulaştı. Hata analizi, kalan hataların ağırlıkla ANLAMSAL uyuşmazlıklarda (marka/model farkı) toplandığını gösterdi — ki elle özellikler bunları yüzeysel olarak yakalayabiliyordu. Bu, daha derin bir anlamsal modele geçme kararının temeliydi.

*(Bu neslin tam kodu geliştirme sürecinde kullanıldı; nihai çıkarım için aşağıdaki 2. yaklaşım — reranker — tek başına yeterlidir.)*

## 6. YAKLAŞIM 2 — BGE CROSS-ENCODER RERANKER (0.743 → 0.805) [NİHAİ]

**Kilit içgörü:** "Arama–ürün alaka" görevi için AMACA ÖZEL eğitilmiş bir cross-encoder reranker vardır. `bge-reranker-v2-m3` (çok dilli, ~568M param) tam bu iş için eğitilmiştir. Sıfır-atış (fine-tune etmeden) kullanıldı.

Ürün "belgesi" = başlık + kategori + marka. Reranker her çifte sürekli bir alaka skoru verir; optimal eşik (-1.5) ile 0/1'e çevrilir.

Gerçekçi doğrulama (Qwen ile etiketlenmiş 5000 çift üzerinde): **AUC = 0.9194 | macro-F1 = 0.8357** (LightGBM'i belirgin biçimde geçti)

> **Not:** `RERANKER_PATH` altındaki model verisinin de notebook'a "Add Input" ile eklendiğinden emin olun (`andreasbis/baai-bge-reranker-v2-m3`), aksi halde bu hücrede de benzer bir `FileNotFoundError`/`OSError` alırsınız.

In [4]:
import os, glob

RERANKER_MODEL_NAME = "baai-bge-reranker-v2-m3"

def find_model_dir(name=RERANKER_MODEL_NAME, root="/kaggle/input"):
    """Model klasörünü /kaggle/input altında otomatik bulur.
    Sabit yol yerine 'config.json' iceren, adinda model adi gecen klasoru arar
    (Kaggle model input'lari genelde farkli versiyon/varyant alt klasorleriyle gelir)."""
    candidates = [
        p for p in glob.glob(f"{root}/**/config.json", recursive=True)
        if name.lower() in p.lower()
    ]
    if not candidates:
        # config.json'da isim gecmeyebilir; sadece klasor adinda ara
        candidates = [
            os.path.dirname(p) for p in glob.glob(f"{root}/**/config.json", recursive=True)
            if name.lower() in os.path.dirname(p).lower()
        ]
        candidates = [os.path.join(c, "config.json") if not c.endswith("config.json") else c for c in candidates]
    if not candidates:
        available = os.listdir(root) if os.path.isdir(root) else "KLASÖR YOK"
        raise FileNotFoundError(
            f"'{name}' modeli '{root}' altinda bulunamadi.\n"
            f"Kaggle notebook editorunde sag panelden 'Add Input' ile "
            f"'andreasbis/baai-bge-reranker-v2-m3' modelini eklediginizden emin olun.\n"
            f"'{root}' icerigi: {available}"
        )
    return os.path.dirname(candidates[0])

RERANKER_PATH = find_model_dir()
THRESHOLD, BATCH, MAX_LEN, CHUNK = -1.5, 128, 256, 200_000

print(f"⏳ Reranker yukleniyor... (yol: {RERANKER_PATH})")
tok = AutoTokenizer.from_pretrained(RERANKER_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_PATH, dtype=torch.float16).to("cuda")
model.eval()
print("✅ Reranker yuklendi.")


⏳ Reranker yukleniyor... (yol: /kaggle/input/models/andreasbis/baai-bge-reranker-v2-m3/transformers/default/1)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker yuklendi.


In [5]:
@torch.no_grad()
def score_pairs(pairs):
    out = np.empty(len(pairs), dtype=np.float32)
    for s in range(0, len(pairs), BATCH):
        b = pairs[s:s + BATCH]
        enc = tok(b, padding=True, truncation=True, max_length=MAX_LEN,
                  return_tensors="pt").to("cuda")
        out[s:s + len(b)] = model(**enc).logits.view(-1).float().cpu().numpy()
    return out


In [6]:
df_test = pl.read_csv(PATHS["submission"])
n_in = df_test.height
if CFG["TERM_ID"] != "term_id":
    df_test = df_test.rename({CFG["TERM_ID"]: "term_id"})
if CFG["ITEM_ID"] != "item_id":
    df_test = df_test.rename({CFG["ITEM_ID"]: "item_id"})
df_test = (df_test
    .join(df_terms, on="term_id", how="left")
    .join(df_items, on="item_id", how="left")
    .with_columns(pl.concat_str([
        pl.col(CFG["TITLE"]).cast(pl.Utf8).fill_null(""),
        pl.col(CFG["CATEGORY"]).cast(pl.Utf8).fill_null(""),
        pl.col(CFG["BRAND"]).cast(pl.Utf8).fill_null(""),
    ], separator=" ").alias("doc")))

ids, scores = [], []
for st in range(0, df_test.height, CHUNK):
    c = df_test.slice(st, min(CHUNK, df_test.height - st))
    q = c[CFG["QUERY"]].cast(pl.Utf8).fill_null("").to_list()
    d = c["doc"].to_list()
    scores.append(score_pairs(list(zip(q, d))))
    ids.append(c["id"].to_numpy())
    print(f"   {st + c.height:,}/{df_test.height:,}")

ids = np.concatenate(ids); scores = np.concatenate(scores)
pred = (scores >= THRESHOLD).astype(int)
submission = pd.DataFrame({"id": ids, "prediction": pred})
assert submission.shape[0] == n_in and submission["id"].is_unique
submission.to_csv("submission.csv", index=False)
print(f"\n🏆 submission.csv: {len(submission):,} satır | eşik {THRESHOLD}")
print(f"   Dağılım: {submission['prediction'].value_counts().to_dict()}")


   200,000/3,359,679
   400,000/3,359,679
   600,000/3,359,679
   800,000/3,359,679
   1,000,000/3,359,679
   1,200,000/3,359,679
   1,400,000/3,359,679
   1,600,000/3,359,679
   1,800,000/3,359,679
   2,000,000/3,359,679
   2,200,000/3,359,679
   2,400,000/3,359,679
   2,600,000/3,359,679
   2,800,000/3,359,679
   3,000,000/3,359,679
   3,200,000/3,359,679
   3,359,679/3,359,679

🏆 submission.csv: 3,359,679 satır | eşik -1.5
   Dağılım: {0: 2429079, 1: 930600}


## 8. ABLASYON — HER ADIMIN KATKISI

| Aşama | Public LB | Katkı |
| :--- | :--- | :--- |
| Başlangıç (TF-IDF + LightGBM) | 0.704 | — |
| + Noktalama temizliği + öznitelikler | 0.731 | +0.025 |
| + Karakter n-gram TF-IDF | 0.737 | +0.006 |
| + Marka uyumu | 0.740 | +0.003 |
| + Cinsiyet uyumu | 0.743 | +0.003 |
| → BGE reranker + optimal eşik (-1.5) | **0.805** | **+0.062** |

**İÇGÖRÜ:** Elle özellikler istikrarlı ama sınırlı katkı sağladı (~+0.04). Asıl sıçrama, DOĞRU ARACI seçmekten geldi (+0.062). "Doğru aleti seçmek, mevcut aleti keskinleştirmekten değerliydi.

## 9. NELER YARAMADI (dürüst eleme)

- **Reranker fine-tuning** (Qwen etiketli ~22k çift): sıfır-atış modeli (0.805), fine-tuned modeli (0.780) GEÇTİ. Hazır BGE zaten güçlüydü; sınırlı/dengesiz verimiz onu iyileştiremedi.
- **Daha büyük model** (Qwen3-Reranker-4B): 2xT4'te skorlama ~101 saat sürerdi; donanıma sığmadı. Büyük LLM-reranker'lar bu donanımda pratik değil.
- **Öznitelik ekleme / stacking:** reranker'ı geçemedi (o sinyalleri zaten içselleştirmiş).
- **LLM-doğrulamasına göre eşik:** LB'yi düşürdü.

**SONUÇ:** Kelime-örtüşmesi tabanı (~0.63) belirgin biçimde aşılarak 0.805'e ulaşıldı. Yolculuğun ayırt edici yanı: her adımı ölçmek, işe yaramayanı dürüstçe elemek ve probleme en uygun aracı bulmak.